# 🌲 Module 16: Geospatial Vector Polygon Extraction & GIS Export

Welcome to **Module 16** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Operational GIS Integration**: Bridge deep learning raster predictions with vector GIS formats.
2. **Raster-to-Vector Polygonization**: Use affine transformation matrices to produce georeferenced polygons.
3. **Douglas-Peucker Simplification**: Eliminate pixel staircase artifacts and optimize polygon storage.
4. **Multi-Format Export**: Generate GeoJSON FeatureCollections, ESRI Shapefiles, and Cloud-Optimized GeoTIFFs (COG).
5. **Field Attribution**: Attach critical law enforcement attributes (Alert ID, Area, Severity, Driver, Centroid Coordinates).

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_16_geospatial_export.vectorizer import RasterPolygonizer
from modules.module_16_geospatial_export.gis_exporter import export_geojson, export_shapefile, export_cog_geotiff

%matplotlib inline
print("✅ Module 16 GIS Export Environment Ready!")

## 1. Load Satellite Scene & Georeferencing Metadata

Inspect affine transform, coordinate reference system (CRS), and raster masks.

In [ ]:
with rasterio.open("../dataset/test/masks/scene_test_001.tif") as sm:
    mask = sm.read(1)
    transform = sm.transform
    crs = sm.crs

print(f"CRS: {crs}")
print(f"Transform: {transform}")
print(f"Mask shape: {mask.shape}")

## 2. Vectorize Raster to GeoJSON Features

Apply Douglas-Peucker simplification and extract attribute metadata.

In [ ]:
vectorizer = RasterPolygonizer(min_area_ha=0.05, simplify_tolerance=0.0001, pixel_res_meters=30.0)
features = vectorizer.polygonize(mask, transform=transform, crs=str(crs))
print(f"Extracted {len(features)} valid vector polygons")

# Preview first polygon
features[0]["properties"]

## 3. Export to GeoJSON, Shapefile, and COG GeoTIFF

Export all GIS formats ready for QGIS, ArcGIS, and Web Map integration.

In [ ]:
out_geojson = "../outputs/module_16/deforestation_alerts.geojson"
export_geojson(features, out_geojson)

out_shp = "../outputs/module_16/shapefile/deforestation_alerts.shp"
export_shapefile(features, out_shp)

out_cog = "../outputs/module_16/deforestation_severity_cog.tif"
export_cog_geotiff(mask, out_cog, transform=transform, crs=str(crs))

print("✅ Successfully exported GeoJSON, ESRI Shapefile, and Cloud-Optimized GeoTIFF!")